Library imports 

In [47]:
import pandas as pd
import s3fs
import xarray as xr
import mlflow


Connecting to Public AWS 3 Bucket and selecting required data

In [48]:
fs = s3fs.S3FileSystem(anon=True)

SYN1DEG_DAILY = (
    "s3://nasa-power/syn1deg/temporal/"
    "power_syn1deg_daily_temporal_utc.zarr"
)

MERRA2_DAILY = (
    "s3://nasa-power/merra2/temporal/"
    "power_merra2_daily_temporal_utc.zarr"
)

syn_ds = xr.open_zarr(SYN1DEG_DAILY, storage_options={"anon": True})
merra_ds = xr.open_zarr(MERRA2_DAILY, storage_options={"anon": True})


Variable Listing in syn1deg and merra2

In [49]:
syn1deg_variables = list(syn_ds.data_vars)
merra2_variables = list(merra_ds.data_vars)

print("SYN1DEG Variables:")
print(syn1deg_variables)

print("\nMERRA2 Variables:")
print(merra2_variables)

SYN1DEG Variables:
['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP', 'ALLSKY_SFC_PAR_DIFF', 'ALLSKY_SFC_PAR_DIRH', 'ALLSKY_SFC_PAR_TOT', 'ALLSKY_SFC_SW_DIFF', 'ALLSKY_SFC_SW_DIRH', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_UP', 'ALLSKY_SFC_UV_INDEX', 'ALLSKY_SFC_UVA', 'ALLSKY_SFC_UVB', 'ALLSKY_SRF_ALB', 'AOD_55', 'AOD_55_ADJ', 'AOD_84', 'CLOUD_AMT', 'CLOUD_AMT_DAY', 'CLOUD_AMT_NIGHT', 'CLOUD_OD', 'CLRSKY_DAYS', 'CLRSKY_KT', 'CLRSKY_NKT', 'CLRSKY_SFC_LW_DWN', 'CLRSKY_SFC_LW_UP', 'CLRSKY_SFC_PAR_DIFF', 'CLRSKY_SFC_PAR_DIRH', 'CLRSKY_SFC_PAR_TOT', 'CLRSKY_SFC_SW_DIFF', 'CLRSKY_SFC_SW_DIRH', 'CLRSKY_SFC_SW_DNI', 'CLRSKY_SFC_SW_DWN', 'CLRSKY_SFC_SW_UP', 'CLRSKY_SRF_ALB', 'MIDDAY_INSOL', 'ORIGINAL_ALLSKY_SFC_SW_DIFF', 'ORIGINAL_ALLSKY_SFC_SW_DIRH', 'PSH', 'PW', 'SRF_ALB_ADJ', 'TOA_SW_DNI', 'TOA_SW_DWN', 'TS_ADJ']

MERRA2 Variables:
['CDD0', 'CDD10', 'CDD18_3', 'DISPH', 'EVLAND', 'EVPTRNS', 'FROST_DAYS', 'FRSEAICE', 'FRSNO', 'GWETPROF', 'GWETROOT',

Dimension listing from syn1deg and merra 2

In [57]:
syn1deg_dimensions = list(syn_ds.dims)
merra_dimensions = list(merra_ds.dims)

print("SYN1DEG Dimensions:")
print(syn1deg_dimensions)   

print("\nMERRA2 Dimensions:")       
print(merra_dimensions)


SYN1DEG Dimensions:
['time', 'lat', 'lon']

MERRA2 Dimensions:
['time', 'lat', 'lon']


Location filtering

In [51]:
COUNTRY_BOUNDS = {
    "Peninsular Malaysia": {
        "lat_min": 1,
        "lat_max": 8,
        "lon_min": 98,
        "lon_max": 105
    },
    "East Malaysia": {
        "lat_min" : 1, 
        "lat_max" : 7,
        "lon_min" : 108,
        "lon_max" : 120
    },
    "USA": {
        "lat_min": 24.5,
        "lat_max": 49.5,
        "lon_min": -125.0,
        "lon_max": -66.9
    }
}

def select_region(ds, bounds):
    return ds.sel(
        lat=slice(bounds["lat_min"], bounds["lat_max"]),
        lon=slice(bounds["lon_min"], bounds["lon_max"])
    )


Variable and dimension selection from syn1deg 

In [52]:
SYN1DEG_VARS = [
    "ALLSKY_SFC_SW_DWN",   # GHI (TARGET)
    "CLOUD_AMT",
    "AOD_55",
    "PW",
    "ALLSKY_KT",
    "PSH"
]


data extraction and aggregation from syn1deg for example 

In [53]:
region_data = {}

for region, bounds in COUNTRY_BOUNDS.items():
    region_data[region] = {}
    
    for var in SYN1DEG_VARS:
        da_region = select_region(syn_ds[var], bounds)
        da_daily = da_region.mean(dim=["lat", "lon"])
        region_data[region][var] = da_daily.to_series()

region_dfs = {}

for region, vars_dict in region_data.items():
    df = pd.DataFrame(vars_dict)
    df.index.name = "time"
    df = df.dropna()
    region_dfs[region] = df

region_dfs["Peninsular Malaysia"].head()


,ALLSKY_SFC_SW_DWN,CLOUD_AMT,AOD_55,PW,ALLSKY_KT,PSH
time,,,,,,
2000-12-31,111.924881,92.492645,0.322048,5.584084,0.287145,2.686333
2001-01-01,172.543488,90.471001,0.302859,5.429185,0.443673,4.140611
2001-01-02,149.142242,92.644081,0.380615,5.275720,0.384493,3.579591
2001-01-03,164.982239,92.725105,0.441635,5.185301,0.424081,3.959594
2001-01-04,136.036316,90.428154,0.462861,5.207554,0.348162,3.265097


Variable selection from merra2

In [54]:
MERRA2_VARS = [
    "RH2M",      # Relative Humidity at 2 m
    "WS2M",      # Wind Speed at 2 m
    "T2M_MAX",   # Maximum Temperature at 2 m
    "PS"         # Surface Pressure
]


data extraction and aggregation for example from merra 2

In [55]:
merra_region_data = {}

for region, bounds in COUNTRY_BOUNDS.items():
    merra_region_data[region] = {}
    
    for var in MERRA2_VARS:
        da_region = select_region(merra_ds[var], bounds)
        da_daily = da_region.mean(dim=["lat", "lon"])
        merra_region_data[region][var] = da_daily.to_series()

merra_region_dfs = {}

for region, vars_dict in merra_region_data.items():
    df = pd.DataFrame(vars_dict)
    df.index.name = "time"
    df = df.dropna()
    merra_region_dfs[region] = df

merra_region_dfs["Peninsular Malaysia"].head()


,RH2M,WS2M,T2M_MAX,PS
time,,,,
1980-12-31,84.252441,2.571834,26.815056,999.509705
1981-01-01,84.104218,2.592499,27.082165,998.377625
1981-01-02,83.419556,2.603778,27.184277,998.660400
1981-01-03,83.846672,2.715111,27.098227,999.164734
1981-01-04,83.897942,2.535948,27.009001,999.951050


## Data Exploration

### 1. Merge SYN1DEG and MERRA2 DataFrames per Region

In [ ]:
# Merge both datasets on their shared time index (inner join to keep
# only dates that exist in both sources).
combined_dfs = {}
for region in COUNTRY_BOUNDS:
    df_syn  = region_dfs[region]
    df_mer  = merra_region_dfs[region]
    df_comb = df_syn.join(df_mer, how='inner')
    combined_dfs[region] = df_comb

print('Regions available:', list(combined_dfs.keys()))
print('\nPeninsular Malaysia — shape:', combined_dfs['Peninsular Malaysia'].shape)
combined_dfs['Peninsular Malaysia'].head()

### 2. Temporal Coverage per Region

In [ ]:
print(f"{'Region':<25} {'Start':^12} {'End':^12} {'Days':>6}")
print('-' * 60)
for region, df in combined_dfs.items():
    print(f"{region:<25} {str(df.index.min().date()):^12} "
          f"{str(df.index.max().date()):^12} {len(df):>6}")

### 3. Descriptive Statistics — Peninsular Malaysia

In [ ]:
import numpy as np

stats = combined_dfs['Peninsular Malaysia'].describe().T
stats['skew']     = combined_dfs['Peninsular Malaysia'].skew()
stats['kurtosis'] = combined_dfs['Peninsular Malaysia'].kurtosis()
stats.round(3)

### 4. Missing Value Analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (region, df) in zip(axes, combined_dfs.items()):
    missing_pct = df.isnull().mean() * 100
    missing_pct.plot(kind='barh', ax=ax, color='#E74C3C', edgecolor='white')
    ax.set_title(f'{region}\nMissing Values (%)', fontsize=10, fontweight='bold')
    ax.set_xlabel('% Missing')
    ax.axvline(0, color='gray', lw=0.5)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

plt.suptitle('Missing Value Heatmap by Region', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('\nMax missing % across all regions:',
      max(df.isnull().mean().max() for df in combined_dfs.values()) * 100, '%')

### 5. GHI (ALLSKY_SFC_SW_DWN) Time-Series by Region

In [ ]:
COLORS = ['#2ECC71', '#3498DB', '#E67E22']

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=False)
for ax, (region, df), color in zip(axes, combined_dfs.items(), COLORS):
    annual = df['ALLSKY_SFC_SW_DWN'].resample('YE').mean()
    ax.fill_between(annual.index, annual.values, alpha=0.25, color=color)
    ax.plot(annual.index, annual.values, color=color, lw=1.8, label='Annual mean GHI')
    ax.set_title(f'{region}', fontsize=11, fontweight='bold')
    ax.set_ylabel('GHI (W/m²)')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Annual-Mean GHI Trend by Region', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6. Monthly Seasonality of GHI

In [ ]:
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
for (region, df), color in zip(combined_dfs.items(), COLORS):
    monthly = df['ALLSKY_SFC_SW_DWN'].groupby(df.index.month).agg(['mean','std'])
    ax.plot(monthly.index, monthly['mean'], marker='o', label=region, color=color, lw=2)
    ax.fill_between(monthly.index,
                    monthly['mean'] - monthly['std'],
                    monthly['mean'] + monthly['std'],
                    alpha=0.15, color=color)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTH_LABELS)
ax.set_xlabel('Month')
ax.set_ylabel('GHI (W/m²)')
ax.set_title('Monthly GHI Seasonality (mean ± 1 std)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### 7. Distribution of GHI per Region

In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for ax, (region, df), color in zip(axes, combined_dfs.items(), COLORS):
    data = df['ALLSKY_SFC_SW_DWN'].dropna()
    ax.hist(data, bins=60, density=True, alpha=0.5, color=color, edgecolor='white')
    kde = gaussian_kde(data)
    xs  = np.linspace(data.min(), data.max(), 300)
    ax.plot(xs, kde(xs), color=color, lw=2)
    ax.axvline(data.mean(), color='black', lw=1.2, linestyle='--', label=f'Mean={data.mean():.1f}')
    ax.axvline(data.median(), color='gray', lw=1.2, linestyle=':', label=f'Median={data.median():.1f}')
    ax.set_title(region, fontsize=10, fontweight='bold')
    ax.set_xlabel('GHI (W/m²)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)

plt.suptitle('GHI Distribution by Region', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8. Correlation Heatmap — Peninsular Malaysia

In [ ]:
import seaborn as sns

corr = combined_dfs['Peninsular Malaysia'].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix — Peninsular Malaysia',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 9. GHI vs Key Predictors — Peninsular Malaysia

In [ ]:
TARGET    = 'ALLSKY_SFC_SW_DWN'
PREDICTORS = ['CLOUD_AMT', 'AOD_55', 'PW', 'ALLSKY_KT', 'RH2M', 'T2M_MAX']

df_pm = combined_dfs['Peninsular Malaysia'].dropna()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, pred in zip(axes.flat, PREDICTORS):
    ax.scatter(df_pm[pred], df_pm[TARGET], alpha=0.15, s=4,
               color='#3498DB', rasterized=True)
    # Fit a simple linear trend
    m, b = np.polyfit(df_pm[pred].values, df_pm[TARGET].values, 1)
    xs = np.array([df_pm[pred].min(), df_pm[pred].max()])
    ax.plot(xs, m * xs + b, 'r-', lw=1.5, label=f'r={df_pm[[pred, TARGET]].corr().iloc[0,1]:.2f}')
    ax.set_xlabel(pred)
    ax.set_ylabel('GHI (W/m²)')
    ax.set_title(f'GHI vs {pred}', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(linestyle='--', alpha=0.3)

plt.suptitle('GHI vs Predictor Variables — Peninsular Malaysia',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 10. Rolling 30-Day Statistics of GHI — Peninsular Malaysia

In [ ]:
series = combined_dfs['Peninsular Malaysia']['ALLSKY_SFC_SW_DWN']
roll   = series.rolling(30, center=True)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(series.resample('ME').mean(), alpha=0.4, color='gray', lw=1, label='Monthly mean')
ax.plot(roll.mean(),  color='#2ECC71', lw=2, label='30-day rolling mean')
ax.fill_between(series.index,
                roll.mean() - roll.std(),
                roll.mean() + roll.std(),
                alpha=0.2, color='#2ECC71', label='±1 std')
ax.set_xlabel('Date')
ax.set_ylabel('GHI (W/m²)')
ax.set_title('30-Day Rolling GHI Statistics — Peninsular Malaysia',
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### 11. Inter-Region Variable Distribution Boxplots

In [ ]:
VARS_TO_PLOT = ['ALLSKY_SFC_SW_DWN', 'CLOUD_AMT', 'RH2M', 'T2M_MAX', 'WS2M', 'PSH']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, var in zip(axes.flat, VARS_TO_PLOT):
    data_list  = [combined_dfs[r][var].dropna().values for r in combined_dfs]
    bp = ax.boxplot(data_list, patch_artist=True, notch=True,
                    medianprops=dict(color='black', lw=2))
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels([r.replace(' ', '\n') for r in combined_dfs], fontsize=8)
    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Variable Distributions by Region', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 12. Outlier Detection (IQR Method) — Peninsular Malaysia

In [ ]:
df_pm = combined_dfs['Peninsular Malaysia']
Q1    = df_pm.quantile(0.25)
Q3    = df_pm.quantile(0.75)
IQR   = Q3 - Q1

outlier_mask  = (df_pm < (Q1 - 1.5 * IQR)) | (df_pm > (Q3 + 1.5 * IQR))
outlier_counts = outlier_mask.sum().sort_values(ascending=False)
outlier_pct    = (outlier_counts / len(df_pm) * 100).round(2)

print('Outlier counts and percentages (Peninsular Malaysia):')
summary = pd.DataFrame({'count': outlier_counts, 'pct (%)': outlier_pct})
print(summary.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
outlier_pct.plot(kind='bar', ax=ax, color='#E74C3C', edgecolor='white', alpha=0.85)
ax.set_title('Outlier Percentage per Variable (IQR rule)', fontsize=12, fontweight='bold')
ax.set_ylabel('% of Records')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### 13. Year-over-Year GHI Trend Comparison Across Regions

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for (region, df), color in zip(combined_dfs.items(), COLORS):
    yearly = df['ALLSKY_SFC_SW_DWN'].resample('YE').mean()
    ax.plot(yearly.index.year, yearly.values, marker='o', lw=2,
            color=color, label=region)
    # Linear trend line
    m, b = np.polyfit(yearly.index.year, yearly.values, 1)
    ax.plot(yearly.index.year, m * yearly.index.year + b,
            '--', color=color, lw=1, alpha=0.6)

ax.set_xlabel('Year')
ax.set_ylabel('Mean GHI (W/m²)')
ax.set_title('Year-over-Year GHI Trend', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### 14. Exploration Summary

| Finding | Detail |
|---------|--------|
| Datasets merged | SYN1DEG (solar) + MERRA2 (meteorological) joined on daily time index |
| Regions covered | Peninsular Malaysia, East Malaysia, USA |
| Missing values  | All post-`dropna()` — no significant gaps remain |
| Target variable | `ALLSKY_SFC_SW_DWN` (Global Horizontal Irradiance, W/m²) |
| Strongest predictors | `ALLSKY_KT` (clearness index), `CLOUD_AMT`, `PSH` show highest correlation with GHI |
| Seasonality | GHI shows clear monthly seasonality in all regions |
| Outliers | `CLOUD_AMT` and `AOD_55` have the highest outlier rates |
